# 01 · Data Cleaning & Exploratory Data Analysis

**Business question.** How does scoring in the IPL behave across seasons, teams, and venues? What is the right summary picture to anchor the rest of this project?

**Inputs.** `data/raw/matches.csv`, `data/raw/deliveries.csv` (downloaded from Kaggle).

**Outputs.**
* Cached cleaned data → `data/processed/`
* Descriptive figures → `reports/figures/`
* Descriptive table → `reports/tables/descriptive_summary.csv`

In [1]:
import sys, os
os.chdir('..')  # work from project root for clean paths
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_processed_or_build
from src.viz import savefig, annotate_bars, PRIMARY, ACCENT, PALETTE

Matplotlib is building the font cache; this may take a moment.


## 1.1 Load & clean

In [2]:
matches, deliveries, innings = load_processed_or_build(force_rebuild=True)
print(f'matches    : {len(matches):,} rows · {matches.shape[1]} cols')
print(f'deliveries : {len(deliveries):,} rows · {deliveries.shape[1]} cols')
print(f'innings    : {len(innings):,} rows  (one per match × inning)')
print(f'seasons    : {sorted(matches["season"].unique())}')

matches    : 1,193 rows · 13 cols
deliveries : 283,678 rows · 19 cols
innings    : 2,412 rows  (one per match × inning)
seasons    : [np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## 1.2 Central tendency & spread of first-innings totals

First-innings totals are the cleanest single summary of an IPL match because the second innings is *target-driven* (chase patterns differ fundamentally from setting patterns).

In [3]:
first_inn = innings[innings['inning'] == 1].copy()
desc = first_inn['innings_total'].describe(percentiles=[.1, .25, .5, .75, .9]).round(2)
extras = pd.Series({
    'mode'     : first_inn['innings_total'].mode().iloc[0],
    'skewness' : stats.skew(first_inn['innings_total']),
    'kurtosis' : stats.kurtosis(first_inn['innings_total']),
    'CV (%)'   : desc['std'] / desc['mean'] * 100,
}).round(3)
summary = pd.concat([desc, extras])
summary.to_csv('reports/tables/descriptive_summary.csv', header=['value'])
summary

count       1193.000
mean         167.360
std           33.360
min           25.000
10%          126.000
25%          147.000
50%          167.000
75%          188.000
90%          209.000
max          287.000
mode         162.000
skewness      -0.037
kurtosis       0.690
CV (%)        19.933
dtype: float64

**Read-out.** Mean ≈ median ≈ mode would indicate a near-symmetric distribution. A skewness inside ±0.5 is approximately symmetric; outside ±1 is clearly skewed. A coefficient of variation under 20% means the league is fairly *consistent* in how matches score.

## 1.3 Distribution of first-innings scores

In [4]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.histplot(first_inn['innings_total'], bins=30, kde=True,
             color=PRIMARY, ax=ax)
ax.axvline(desc['mean'],   color=ACCENT, linestyle='--', lw=2,
           label=f"Mean = {desc['mean']:.0f}")
ax.axvline(desc['50%'],    color='#27ae60', linestyle='--', lw=2,
           label=f"Median = {desc['50%']:.0f}")
ax.set_title('Distribution of First-Innings Totals (IPL)')
ax.set_xlabel('First-innings total (runs)')
ax.set_ylabel('Number of matches')
ax.legend()
savefig('eda_01_first_innings_distribution.png')
plt.show()

## 1.4 Team-by-team first-innings totals (top 8 by matches played)

In [5]:
top_teams = first_inn['batting_team'].value_counts().head(8).index.tolist()
sub = first_inn[first_inn['batting_team'].isin(top_teams)]
team_order = sub.groupby('batting_team')['innings_total'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.boxplot(data=sub, x='batting_team', y='innings_total',
            order=team_order, palette=PALETTE, ax=ax)
plt.xticks(rotation=20, ha='right')
ax.set_title('First-Innings Totals by Franchise')
ax.set_xlabel('')
ax.set_ylabel('Runs')
savefig('eda_02_team_boxplot.png')
plt.show()

/var/folders/zc/kthpnxfd60g1x98xxryqmd4r0000gn/T/ipykernel_92044/1889591898.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sub, x='batting_team', y='innings_total',
/var/folders/zc/kthpnxfd60g1x98xxryqmd4r0000gn/T/ipykernel_92044/1889591898.py:6: UserWarning: 
The palette list has fewer values (6) than needed (8) and will cycle, which may produce an uninterpretable plot.
  sns.boxplot(data=sub, x='batting_team', y='innings_total',


## 1.5 League-wide scoring trend across seasons

In [6]:
season_stats = first_inn.groupby('season')['innings_total'].agg(['mean', 'std', 'count']).reset_index()
season_stats['season_int'] = pd.to_numeric(season_stats['season'], errors='coerce')
season_stats = season_stats.dropna(subset=['season_int']).sort_values('season_int')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(season_stats['season_int'], season_stats['mean'],
        marker='o', color=PRIMARY, linewidth=2, label='Mean')
ax.fill_between(season_stats['season_int'],
                season_stats['mean'] - season_stats['std'],
                season_stats['mean'] + season_stats['std'],
                color=PRIMARY, alpha=0.15, label='±1 SD')
ax.set_title('First-Innings Score Across Seasons')
ax.set_xlabel('Season')
ax.set_ylabel('Average runs')
ax.legend()
savefig('eda_03_season_trend.png')
plt.show()

## 1.6 Highest- and lowest-scoring venues

In [7]:
venue_avg = (first_inn.groupby('venue')['innings_total']
             .agg(['mean', 'count']).reset_index())
venue_avg = venue_avg[venue_avg['count'] >= 15].sort_values('mean', ascending=False)
venue_avg.head(10).to_csv('reports/tables/top_venues.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 6))
top = venue_avg.head(10)
sns.barplot(data=top, y='venue', x='mean', palette='viridis', ax=ax)
for i, (_, row) in enumerate(top.iterrows()):
    ax.text(row['mean'] + 1, i, f"{row['mean']:.0f} (n={row['count']})",
            va='center', fontsize=9)
ax.set_title('Top 10 Highest-Scoring Venues (min. 15 matches)')
ax.set_xlabel('Average first-innings runs')
ax.set_ylabel('')
savefig('eda_04_top_venues.png')
plt.show()

/var/folders/zc/kthpnxfd60g1x98xxryqmd4r0000gn/T/ipykernel_92044/1482408148.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top, y='venue', x='mean', palette='viridis', ax=ax)


## 1.7 What we learned

* The first-innings total in the IPL behaves like an approximately symmetric, mound-shaped distribution centred around the league mean — a clean candidate for Normal modelling in the next notebook.
* Different franchises have visibly different median totals and different *consistency* (boxplot IQR).
* The league-wide scoring trend is informative when telling the narrative — broadcast and franchise economics depend on it.
* Venue effects are real and material, justifying the use of venue features in the prediction model (Notebook 04).